# Shots-Form Poisson Model\n\nWie `form_poisson`, aber die Formkurve basiert auf **Schüssen aufs Tor** (`hst`/`ast`, bisher ungenutzte Spalten) statt auf Toren/Punkten - ein rauschärmerer Qualitätsindikator (ähnlich Expected Goals), da Tore stark vom Zufall beim Abschluss abhängen. Trainiert auf 21/22-24/25, getestet Spieltag für Spieltag gegen 25/26, Formkurve wird pro Spieltag aktualisiert.\n\nFür den Modellvergleich siehe `model_comparison.ipynb`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.csv_source import CSVSource
from src.data.loader import DataLoader
from src.models.shots_form_poisson.features import build_design_matrix, compute_current_form
from src.models.shots_form_poisson.predict import predict_score
from src.models.poisson_regressor import PoissonRegressor
from src.models.scoring import kicktipp_points

FORM_WINDOW = 5

In [ ]:
DATA_DIR = Path.cwd().parent / "data" / "raw"
TRAIN_SEASONS = ["season-2122.csv", "season-2223.csv", "season-2324.csv", "season-2425.csv"]

frames = []
for filename in TRAIN_SEASONS:
    loader = DataLoader(CSVSource(DATA_DIR / filename))
    frames.append(loader.load())

train_df = pd.concat(frames, ignore_index=True)
train_df.shape

In [ ]:
X, y, team_index = build_design_matrix(train_df, form_window=FORM_WINDOW)
X.shape, y.shape, len(team_index)

In [ ]:
model = PoissonRegressor(learning_rate=0.1, n_iterations=5000)
model.fit(X, y)

In [ ]:
plt.plot(model.loss_history_)
plt.xlabel("Iteration")
plt.ylabel("Loss (mean negative log-likelihood)")
plt.title("Training loss")
plt.show()

print(f"Form-Koeffizienten (shots_for, shots_against): {model.coef_[-2:]}")

In [ ]:
n_teams = len(team_index)
attack = model.coef_[1 : 1 + n_teams]
defense = model.coef_[1 + n_teams : 1 + 2 * n_teams]
home_advantage = model.coef_[0]

ratings = pd.DataFrame({
    "team": list(team_index.keys()),
    "attack": attack,
    "defense": defense,
}).sort_values("attack", ascending=False)

print(f"Home advantage (log-scale): {home_advantage:.3f}")
ratings

## Vergleich mit Saison 25/26

In [ ]:
test_df = DataLoader(CSVSource(DATA_DIR / "season-2526.csv")).load()
test_df = test_df.sort_values("date", kind="stable").reset_index(drop=True)
test_df["matchday"] = test_df.index // 9 + 1
test_df.shape

In [ ]:
history_df = train_df.copy()
results = []

for matchday in range(1, test_df["matchday"].max() + 1):
    matchday_matches = test_df[test_df["matchday"] == matchday]
    current_form = compute_current_form(history_df, form_window=FORM_WINDOW)

    for match in matchday_matches.itertuples():
        predicted = predict_score(model, match.hometeam, match.awayteam, team_index, current_form)
        actual = (int(match.fthg), int(match.ftag))
        results.append({
            "matchday": matchday,
            "home": match.hometeam,
            "away": match.awayteam,
            "predicted": f"{predicted[0]}:{predicted[1]}",
            "actual": f"{actual[0]}:{actual[1]}",
            "points": kicktipp_points(predicted, actual),
        })

    history_df = pd.concat([history_df, matchday_matches], ignore_index=True)

results_df = pd.DataFrame(results)
results_df.head(9)

In [ ]:
points_per_matchday = results_df.groupby("matchday")["points"].sum()

plt.plot(points_per_matchday.index, points_per_matchday.values, marker="o")
plt.xlabel("Spieltag")
plt.ylabel("Punkte")
plt.title("Kicktipp-Punkte pro Spieltag (Saison 25/26, Schuss-Form-Modell)")
plt.show()

print(f"Gesamtpunkte: {results_df['points'].sum()} ({results_df['points'].mean():.2f} im Schnitt pro Spiel)")
print("Verteilung:")
print(results_df["points"].value_counts().sort_index(ascending=False))

In [ ]:
def categorize(points: int) -> str:
    if points == 4:
        return "exakt"
    if points in (2, 3):
        return "tendenz"
    return "falsch"

results_df["category"] = results_df["points"].apply(categorize)

matchday_summary = (
    results_df
    .groupby(["matchday", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["exakt", "tendenz", "falsch"], fill_value=0)
)
matchday_summary